# IndexTTS2 — Mitra da Dhaba (GPU)

Runs the IndexTTS2 checkpoint from the Hugging Face bucket
[`ayannnn/mitra_da_dhaba`](https://huggingface.co/buckets/ayannnn/mitra_da_dhaba) on a Colab GPU.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better), then `Runtime → Run all`.

1. Setup (~3–5 min, once per session)
2. Download weights (~5.9 GB, ~1–2 min)
3. Pick a reference voice (upload your own, or use a demo voice)
4. Generate — edit the text/emotion form and re-run just that cell
5. *(Optional)* Web UI with a public link, for interactive use

In [ ]:
#@title 0. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: switch runtime to T4 GPU before continuing"

In [ ]:
#@title 1. Setup (clone IndexTTS + install deps)
%cd /content
INDEXTTS_COMMIT = "d9e41aac89fd00b3d71497fddb287b7f24613712"  # version this notebook was tested against
!test -d index-tts || GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/index-tts/index-tts.git
%cd /content/index-tts
!git -c advice.detachedHead=false checkout -q {INDEXTTS_COMMIT}
!pip install -q uv
!uv sync --extra webui 2>&1 | tail -n 3
!wget -q -O /content/tts.py https://raw.githubusercontent.com/ayanisone/11/claude/optimistic-knuth-m9vqi7/scripts/tts.py
!.venv/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())"

In [ ]:
#@title 2. Download weights from the bucket
import json, os, subprocess, urllib.request
BUCKET = "ayannnn/mitra_da_dhaba"
DEST = "/content/index-tts/checkpoints"
files = [f for f in json.load(urllib.request.urlopen(
    f"https://huggingface.co/api/buckets/{BUCKET}/tree?recursive=true")) if f["type"] == "file"]
for f in files:
    path = os.path.join(DEST, f["path"])
    if os.path.exists(path) and os.path.getsize(path) == f["size"]:
        continue
    os.makedirs(os.path.dirname(path), exist_ok=True)
    print("downloading", f["path"], f"{f['size'] / 1e6:.1f} MB")
    subprocess.run(["curl", "-sSfL", "--retry", "4", "-o", path,
                    f"https://huggingface.co/buckets/{BUCKET}/resolve/{f['path']}"], check=True)
bad = [f["path"] for f in files if os.path.getsize(os.path.join(DEST, f["path"])) != f["size"]]
print("all", len(files), "files OK" if not bad else f"SIZE MISMATCH: {bad}")

In [ ]:
#@title 3. Reference voice
#@markdown Tick **upload** to use your own clip: 3–15 s, one speaker, dry (no music/reverb), low noise.
#@markdown Otherwise a demo voice from the official IndexTTS2 Space is used.
upload = False #@param {type:"boolean"}
demo_voice = "voice_01" #@param ["voice_01", "voice_02", "voice_03", "voice_04", "voice_05", "voice_06", "voice_07", "voice_08", "voice_09", "voice_11", "voice_12"]
import os, shutil, subprocess
from IPython.display import Audio, display
os.makedirs("/content/voices", exist_ok=True)
if upload:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    VOICE = f"/content/voices/{name}"
    shutil.move(name, VOICE)
else:
    VOICE = f"/content/voices/{demo_voice}.wav"
    if not os.path.exists(VOICE):
        subprocess.run(["curl", "-sSfL", "-o", VOICE,
            f"https://huggingface.co/spaces/IndexTeam/IndexTTS-2-Demo/resolve/main/examples/{demo_voice}.wav"], check=True)
print("voice:", VOICE)
display(Audio(VOICE))

In [ ]:
#@title 4. Generate
text = "Welcome to Mitra da Dhaba. The tandoor is hot and the chai is on the house." #@param {type:"string"}
out_name = "line_01" #@param {type:"string"}
#@markdown **Emotion mode** — `from voice` = copy the delivery of the reference clip;
#@markdown `vector` = use the sliders; `text description` = describe it in words (e.g. "excited, welcoming").
#@markdown `emo_alpha` scales the emotion strength in `vector` / `text description` modes (ignored for `from voice`).
emotion_mode = "from voice" #@param ["from voice", "vector", "text description"]
emo_alpha = 0.8 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
emo_text = "warm and welcoming" #@param {type:"string"}
happy = 0.6 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
angry = 0.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
sad = 0.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
afraid = 0.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
disgusted = 0.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
melancholic = 0.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
surprised = 0.0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
calm = 0.2 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
ceiling_db = -1.0 #@param {type:"number"}

import json, os
from IPython.display import Audio, display
os.makedirs("/content/outputs", exist_ok=True)
job = {"voice": VOICE, "text": text, "out": f"/content/outputs/{out_name}.wav", "emo_alpha": emo_alpha}
if emotion_mode == "vector":
    job["emo_vector"] = [happy, angry, sad, afraid, disgusted, melancholic, surprised, calm]
elif emotion_mode == "text description":
    job["emo_text"] = emo_text
with open("/content/jobs.jsonl", "w") as f:
    f.write(json.dumps(job) + "\n")
%cd /content/index-tts
!.venv/bin/python /content/tts.py --jobs /content/jobs.jsonl --ceiling-db {ceiling_db} 2>&1 | grep -E "^>> (device|model loaded|/content)|RTF|Error|Traceback"
display(Audio(job["out"]))

In [ ]:
#@title 5. Download all outputs as a zip
!cd /content && zip -qr outputs.zip outputs
from google.colab import files
files.download("/content/outputs.zip")

## Optional: interactive Web UI

Starts the official IndexTTS Gradio app and prints a public `*.gradio.live` link.
The model stays loaded, so every generation after the first is fast. Stop the cell to shut it down.

In [ ]:
#@title 6. Launch Web UI
%cd /content/index-tts
!.venv/bin/python webui.py --version 2 --model_dir ./checkpoints --fp16 --share